In [171]:
import h5py
from sipyco import pyon
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
import os as os
import sys
sys.path.append('/home/ae19663/artiq')  # Adjust this path to the parent directory of 'repository'
from repository.imaging.processor import AbsImage  # noqa: E402

def get_override_args(filename, hide_defaults=True):
    vals = {}

    f1 = h5py.File(filename, "r")
    args = pyon.decode(f1["expid"][()])["arguments"]
    for k, v in args.items():
        if k == "ndscan_params":
            params = pyon.decode(args["ndscan_params"])
            overrides = params["overrides"]
            for k, v in overrides.items():
                spec = params["schemata"][k]["spec"]
                val = v[0]["value"]
                default = type(val)(params["schemata"][k]["default"])
                if hide_defaults and val == default:
                    continue
                vals[k] = [val / spec.get("scale", 1), spec.get("unit", "")]
        else:
            vals[k] = [v, ""]
    return vals

vals = get_override_args("../results/2025-07-08/21/000006529-AbsorptionImageExpFrag.h5")


for k, (v, unit) in vals.items():
    print(f"{k} = {v} {unit}")


absorption_image.AbsorptionImageExpFrag.exposure_time = 0.2 ms
absorption_image.AbsorptionImageExpFrag.expansion_time = 10.0 ms
absorption_image.AbsorptionImageExpFrag.do_pgc = 0.0 
absorption_image.AbsorptionImageExpFrag.do_odt = 0.0 
repository.fragments.mot.MOT.ODT_duration = 10.0 ms
repository.fragments.mot.MOT.CMOT_detuning = 8.0 Γ
repository.fragments.mot.MOT.CMOT_duration = 10.0 ms
repository.fragments.mot.MOT.CMOT_settle_time = 0.0 ms
repository.fragments.mot.MOT.loading_time = 30.0 s
repository.fragments.mot.MOT.ODT_overlap_time = 10.0 ms


In [ ]:
def sorted_files(directory):
    """
    Sort files in the given directory based on the numeric part of their names.
    """
    files = os.listdir(directory)
    file_num = []
    file_name = []
    
    for i in files:
        # Split the file name into two parts and sort by the numeric part
        file_num.append(i.split("-")[0])
        file_name.append(i.split("-")[1])
    
    # Sort the files based on the numeric part
    file_num, file_name = zip(*sorted(zip(file_num, file_name)))
    
    # Recombine the file number and file name into a single list
    sorted_files_list = [f"{num}-{name}" for num, name in zip(file_num, file_name)]
    
    # Directory list for the files using the sorted file names
    files_directory = [os.path.join(directory, f) for f in sorted_files_list if f.endswith(".h5")]
    
    return files_directory


files_directory = sorted_files("../results/2025-07-08/21/")
print(files_directory)


# Extracting the values and the units from the files
f1 = h5py.File(files_directory[0], "r")

for i in f1["datasets"].keys():
    print(i)
    #print(type(i))

absimg = AbsImage(
    data=f1["datasets"]["Images.absorption.TOF"][()],
    ref=f1["datasets"]["Images.absorption.REF"][()],
    bg=f1["datasets"]["Images.absorption.BG"][()],
    magnification=0.25,  # Set default magnification
)


absimg.all_info()
absimg.plot()


# check if set of data files are from the scan or not because in the scan all names of the arguments will be same
def is_scan_data(files_directory):
    if not files_directory:
        return False
    first_file = files_directory[0]
    first_args = get_override_args(first_file)
    for file in files_directory[1:]:
        args = get_override_args(file)
        if args != first_args:
            return False
    return True

a= is_scan_data(files_directory)
print(f"Is scan data: {a}")


# check what parameter is sweeped in the scan
def get_sweeped_parameter(files_directory):


['../results/2025-07-08/21/000006517-AbsorptionImageExpFrag.h5', '../results/2025-07-08/21/000006518-AbsorptionImageExpFrag.h5', '../results/2025-07-08/21/000006519-AbsorptionImageExpFrag.h5', '../results/2025-07-08/21/000006520-AbsorptionImageExpFrag.h5', '../results/2025-07-08/21/000006521-AbsorptionImageExpFrag.h5', '../results/2025-07-08/21/000006522-AbsorptionImageExpFrag.h5', '../results/2025-07-08/21/000006523-AbsorptionImageExpFrag.h5', '../results/2025-07-08/21/000006524-AbsorptionImageExpFrag.h5', '../results/2025-07-08/21/000006525-AbsorptionImageExpFrag.h5', '../results/2025-07-08/21/000006526-AbsorptionImageExpFrag.h5', '../results/2025-07-08/21/000006527-AbsorptionImageExpFrag.h5', '../results/2025-07-08/21/000006528-AbsorptionImageExpFrag.h5', '../results/2025-07-08/21/000006529-AbsorptionImageExpFrag.h5']
Images.All.0
Images.All.1
Images.All.2
Images.Latest_image
Images.absorption.BG
Images.absorption.REF
Images.absorption.TOF
Images.absorption.expansion_time
Images.abs

In [ ]:
#plotting the data from the scan